# Agente de reunião pt-BR com RL (ART + RULER) — Google Colab

Réplica do [ART-E](https://colab.research.google.com/github/openpipe/art-notebooks/blob/main/examples/art-e.ipynb) adaptada para um **assistente de reuniões em português**: o agente busca segmentos de uma transcrição via ferramentas e responde perguntas (tarefas, decisões, riscos). Treino via GRPO com **RULER** — sem dados rotulados.

**Pipeline:** dataset sintético pt-BR (incluso) → *(opcional)* expansão via Gemini Flash-Lite → rollout multi-turno com tool-calling → RULER → GRPO → validação.

**Antes de rodar:**
1. `Ambiente de execução > Alterar tipo de ambiente > GPU` (T4 grátis serve para o modelo 4B; A100 para o Qwen3.6-27B).
2. Configure os secrets no Colab (ícone de chave 🔑 na barra lateral): `OPENAI_API_KEY` (obrigatório — juiz RULER) e `GEMINI_API_KEY` (opcional — expansão do dataset).

⚠️ Smoke test honesto: com o dataset base (5 cenários de treino), rode poucos steps — o modelo memoriza rápido. Para treino real, use a célula de expansão sintética (100+ cenários).

In [ ]:
# GPU disponível?
!nvidia-smi | head -12

In [ ]:
# LocalBackend usa um servidor vLLM para inferência/rollouts.
# Em alguns ambientes Colab, openpipe-art[backend] não deixa `vllm`
# disponível no kernel principal; instale explicitamente.
%pip install -q -U uv
%pip install -q -U "openpipe-art[backend]" "vllm" openai langdetect

# Depois desta célula, se o Colab avisar que precisa reiniciar, use:
# Runtime/Ambiente de execução > Restart session/Reiniciar sessão.


In [ ]:
import os
from getpass import getpass

# Lê dos secrets do Colab; cai para prompt interativo fora do Colab.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    try:
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    except Exception:
        print("GEMINI_API_KEY ausente — a expansão sintética será pulada.")
except ImportError:
    os.environ.setdefault("OPENAI_API_KEY", getpass("OPENAI_API_KEY: "))

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY é obrigatória (juiz RULER)"

## 1. Dataset sintético pt-BR (incluso)

8 reuniões fictícias (5 treino, 3 validação), incluindo um **cenário de controle sem decisões** — ensina o modelo a não inventar tarefas. O RULER não precisa de rótulos; o campo `expected` alimenta só o juiz de validação.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Segment:
    idx: int
    speaker: str
    text: str

@dataclass
class MeetingScenario:
    id: str
    title: str
    date: str
    segments: list
    question: str
    expected: str = ""
    split: str = "train"

def _seg(rows):
    return [Segment(i, sp, tx) for i, (sp, tx) in enumerate(rows)]

SCENARIOS = [
    MeetingScenario(
        id="planning-q3", title="Planejamento Q3 — Produto", date="2026-06-10",
        segments=_seg([
            ("Ana", "Bom dia pessoal. Hoje precisamos fechar o escopo do Q3."),
            ("Bruno", "A prioridade número um continua sendo o app de desktop, certo?"),
            ("Ana", "Sim. O desktop tem que sair até o fim de julho, é compromisso com o cliente."),
            ("Carla", "Eu assumo a parte de captura de áudio do desktop então."),
            ("Bruno", "E eu fico com a integração do Teams. Fecho a POC até dia 20."),
            ("Ana", "Perfeito. A Carla entrega captura até 25 de julho e o Bruno a POC do Teams até 20 de junho."),
            ("Carla", "Combinado. Alguma dependência externa que eu precise saber?"),
            ("Ana", "Só o SDK de captura, mas isso já está aprovado pelo jurídico."),
        ]),
        question="Quais são as tarefas atribuídas e seus responsáveis e prazos?",
        expected="Carla: captura de áudio do desktop até 25/07. Bruno: POC de integração do Teams até 20/06.",
        split="train",
    ),
    MeetingScenario(
        id="incident-review", title="Revisão de Incidente — Fila de Transcrição", date="2026-06-12",
        segments=_seg([
            ("Diego", "A fila de transcrição travou ontem às 14h por duas horas."),
            ("Elena", "A causa foi o pico de reuniões simultâneas, o worker não escalou."),
            ("Diego", "Precisamos de auto-scaling na fila até o fim da semana."),
            ("Elena", "Eu configuro o auto-scaling. Sexta no máximo."),
            ("Diego", "E vamos adicionar um alerta de profundidade de fila também."),
            ("Fábio", "Eu crio o alerta de fila, posso entregar na quinta."),
            ("Diego", "Ótimo. Nenhum dado de cliente foi perdido, só houve atraso."),
        ]),
        question="O que causou o incidente e quais ações foram definidas para evitar recorrência?",
        expected="Causa: pico de reuniões simultâneas sem auto-scaling do worker. Ações: Elena configura auto-scaling até sexta; Fábio cria alerta de profundidade de fila até quinta.",
        split="train",
    ),
    MeetingScenario(
        id="sales-sync", title="Sync Comercial — Conta Northwind", date="2026-06-15",
        segments=_seg([
            ("Gabriela", "A Northwind quer expandir de 50 para 300 licenças."),
            ("Hugo", "Ótimo. Eles pediram algum requisito novo?"),
            ("Gabriela", "Sim, isolamento de dados por região e retenção de 90 dias."),
            ("Hugo", "Retenção configurável já temos. Isolamento por região é roadmap."),
            ("Gabriela", "Eles topam esperar o isolamento se tivermos data marcada."),
            ("Hugo", "Vou levantar o esforço de isolamento por região com o time de dados."),
            ("Gabriela", "Fecho a proposta de 300 licenças até quarta que vem."),
        ]),
        question="Quais requisitos o cliente pediu e qual é o próximo passo de cada pessoa?",
        expected="Requisitos: isolamento de dados por região e retenção de 90 dias. Próximos passos: Hugo levanta o esforço de isolamento por região; Gabriela fecha a proposta de 300 licenças até quarta.",
        split="train",
    ),
    MeetingScenario(
        id="design-review", title="Design Review — Bot do Teams", date="2026-06-18",
        segments=_seg([
            ("Iara", "O @TagBot precisa responder menção em canal e em DM."),
            ("João", "Em canal, só responde se for explicitamente mencionado, senão vira spam."),
            ("Iara", "Concordo. E em DM responde sempre."),
            ("João", "A latência alvo é abaixo de 3 segundos para a primeira resposta."),
            ("Iara", "Eu escrevo a spec da máquina de estados do bot até segunda."),
            ("João", "Eu prototipo o handler de menção até quarta."),
        ]),
        question="Quais decisões de comportamento do bot foram tomadas e quem entrega o quê?",
        expected="Decisões: responde em canal só quando mencionado, sempre em DM, latência alvo <3s. Iara escreve a spec até segunda; João prototipa o handler de menção até quarta.",
        split="train",
    ),
    MeetingScenario(
        id="retention-policy", title="Política de Retenção e Consentimento", date="2026-06-20",
        segments=_seg([
            ("Lara", "Todo áudio gravado precisa de consentimento explícito do organizador."),
            ("Marcos", "E a retenção padrão do áudio bruto?"),
            ("Lara", "Trinta dias para o áudio, um ano para a transcrição."),
            ("Marcos", "Cada tenant pode encurtar, mas não estender além disso."),
            ("Lara", "Exato. Eu documento a política até sexta."),
            ("Marcos", "Eu implemento o job de expiração de áudio até o fim do mês."),
        ]),
        question="Quais são as regras de consentimento e retenção definidas?",
        expected="Consentimento explícito do organizador para gravar áudio. Retenção: 30 dias para áudio bruto, 1 ano para transcrição; tenant pode encurtar mas não estender. Lara documenta até sexta; Marcos implementa o job de expiração até o fim do mês.",
        split="train",
    ),
    MeetingScenario(
        id="hiring-plan", title="Plano de Contratação — Time de IA", date="2026-06-22",
        segments=_seg([
            ("Núbia", "Precisamos de dois engenheiros de ML no Q3."),
            ("Otávio", "Um sênior para o pipeline de RL e um pleno para avaliação."),
            ("Núbia", "Eu abro as duas vagas até quinta."),
            ("Otávio", "Eu preparo o desafio técnico de RL até a semana que vem."),
            ("Núbia", "Meta é fechar as contratações até o fim de agosto."),
        ]),
        question="Quantas vagas, com quais perfis, e quais são as ações e prazos?",
        expected="Duas vagas: um ML sênior (pipeline de RL) e um pleno (avaliação). Núbia abre as vagas até quinta; Otávio prepara o desafio técnico até a semana seguinte; meta de fechar até fim de agosto.",
        split="val",
    ),
    MeetingScenario(
        id="pricing-debate", title="Debate de Precificação", date="2026-06-24",
        segments=_seg([
            ("Paula", "O plano por assento não está funcionando para clientes grandes."),
            ("Rui", "Proponho cobrança híbrida: assento mais consumo de minutos transcritos."),
            ("Paula", "Risco de assustar o cliente com conta variável."),
            ("Rui", "Colocamos um teto mensal para dar previsibilidade."),
            ("Paula", "Gostei. Eu modelo três cenários de receita até terça."),
            ("Rui", "Eu levanto o custo real de transcrição por minuto até segunda."),
        ]),
        question="Qual mudança de precificação foi proposta e quais tarefas ficaram definidas?",
        expected="Proposta: cobrança híbrida (assento + consumo de minutos transcritos) com teto mensal. Paula modela três cenários de receita até terça; Rui levanta o custo real por minuto até segunda.",
        split="val",
    ),
    MeetingScenario(
        id="empty-decisions", title="Bate-papo sem decisões", date="2026-06-25",
        segments=_seg([
            ("Sofia", "Só queria alinhar como todos estão, sem pauta fechada hoje."),
            ("Tiago", "Por mim tudo tranquilo, semana calma."),
            ("Sofia", "Ótimo. Então não temos nada a decidir, bom descanso a todos."),
        ]),
        question="Quais tarefas e responsáveis foram definidos nesta reunião?",
        expected="Nenhuma tarefa foi definida; a reunião não teve decisões.",
        split="val",
    ),
]

def train_scenarios():
    return [s for s in SCENARIOS if s.split == "train"]

def val_scenarios():
    return [s for s in SCENARIOS if s.split == "val"]

print(f"{len(train_scenarios())} cenários de treino, {len(val_scenarios())} de validação")

## 2. (Opcional) Expandir o dataset com o Gemini 2.5 Flash-Lite

O professor mais barato de Google/OpenAI (US$0,10/1M in, US$0,40/1M out). ~US$0,25 por 200 cenários. Pula automaticamente se `GEMINI_API_KEY` não estiver configurada.

⚠️ Termos de uso: provedores restringem treinar modelos concorrentes com seus outputs — leia os termos antes de escalar; alternativa sem restrição é um professor open-weight.

In [ ]:
import asyncio, json, random
from openai import AsyncOpenAI

N_EXTRA = 50  # suba para 200+ em treino real

if os.environ.get("GEMINI_API_KEY"):
    teacher = AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.environ["GEMINI_API_KEY"],
    )
    DOMAINS = [
        "planejamento de produto", "revisão de incidente de produção",
        "reunião comercial com cliente", "design review de software",
        "compliance e privacidade de dados", "contratação e time",
        "precificação e finanças", "retrospectiva de sprint",
    ]
    QTYPES = [
        "tarefas atribuídas com responsáveis e prazos",
        "decisões tomadas e seus motivos",
        "riscos ou bloqueios levantados",
        "próximos passos por pessoa",
    ]
    GEN_PROMPT = (
        'Gere uma reunião de trabalho FICTÍCIA e realista em português do Brasil.\n'
        'Domínio: {domain}\nA reunião {control} conter tarefas/decisões concretas.\n'
        'Número de segmentos de fala: entre 25 e 60 (segmentos curtos, 1-3 frases, '
        'como saída real de diarização; inclua digressões e small talk).\n'
        'Depois formule UMA pergunta sobre: {qtype}. Por fim escreva a resposta ideal '
        '(objetiva, fiel à transcrição; se não houver tarefas/decisões, diga isso).\n'
        'Responda APENAS com JSON válido: {{"title": "...", "date": "AAAA-MM-DD", '
        '"segments": [{{"speaker": "Nome", "text": "..."}}], "question": "...", "expected": "..."}}'
    )

    async def gen_one(i):
        rng = random.Random(i)
        is_control = rng.random() < 0.15
        prompt = GEN_PROMPT.format(
            domain=rng.choice(DOMAINS),
            control="NÃO deve" if is_control else "deve",
            qtype=rng.choice(QTYPES),
        )
        try:
            resp = await teacher.chat.completions.create(
                model="gemini-2.5-flash-lite",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                max_tokens=4096, temperature=1.0,
            )
            d = json.loads(resp.choices[0].message.content or "")
            return MeetingScenario(
                id=f"gen-{i:04d}", title=d["title"], date=d["date"],
                segments=_seg([(s["speaker"], s["text"]) for s in d["segments"]]),
                question=d["question"], expected=d["expected"],
                split="train" if i % 10 else "val",  # ~10% para validação
            )
        except Exception:
            return None

    extra = [s for s in await asyncio.gather(*(gen_one(i) for i in range(N_EXTRA))) if s]
    SCENARIOS.extend(extra)
    print(f"+{len(extra)} cenários gerados -> {len(train_scenarios())} treino / {len(val_scenarios())} val")
    print("Inspecione algumas amostras antes de treinar:")
    if extra:
        s = extra[0]
        print(f"  [{s.id}] {s.title} — {len(s.segments)} segmentos\n  Q: {s.question}\n  E: {s.expected[:120]}")
else:
    print("GEMINI_API_KEY ausente — usando só os 8 cenários base (smoke test).")

## 3. Ferramentas do agente e helpers

Análogo direto do ART-E: `search_transcript` ↔ `search_inbox`, `read_segment` ↔ `read_email`, `return_final_answer` idem. Inclui o helper do workaround multi-turn do Qwen 3.x.

In [ ]:
import re

TOOLS = [
    {"type": "function", "function": {
        "name": "search_transcript",
        "description": "Busca segmentos da transcrição que contenham as palavras-chave.",
        "parameters": {"type": "object", "properties": {
            "keywords": {"type": "array", "items": {"type": "string"}}},
            "required": ["keywords"]}}},
    {"type": "function", "function": {
        "name": "read_segment",
        "description": "Lê o texto completo de um segmento pelo seu índice.",
        "parameters": {"type": "object", "properties": {
            "segment_idx": {"type": "integer"}}, "required": ["segment_idx"]}}},
    {"type": "function", "function": {
        "name": "return_final_answer",
        "description": "Devolve a resposta final ao usuário e encerra a tarefa.",
        "parameters": {"type": "object", "properties": {
            "answer": {"type": "string"},
            "segment_refs": {"type": "array", "items": {"type": "integer"}}},
            "required": ["answer"]}}},
]

def _search(scenario, keywords):
    kws = [k.lower() for k in keywords]
    hits = [{"segment_idx": s.idx, "speaker": s.speaker, "snippet": s.text[:80]}
            for s in scenario.segments
            if any(k in s.text.lower() or k in s.speaker.lower() for k in kws)]
    return json.dumps({"results": hits[:10]}, ensure_ascii=False)

def _read(scenario, idx):
    for s in scenario.segments:
        if s.idx == idx:
            return json.dumps({"segment_idx": s.idx, "speaker": s.speaker, "text": s.text}, ensure_ascii=False)
    return json.dumps({"error": f"segmento {idx} não existe"}, ensure_ascii=False)

def _strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.DOTALL).strip()

def _choice_as_context(choice):
    msg = choice.message
    out = {"role": "assistant", "content": _strip_think(msg.content or "")}
    if msg.tool_calls:
        out["tool_calls"] = [{"id": tc.id, "type": "function",
                              "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                             for tc in msg.tool_calls]
    return out

SYSTEM_PROMPT = (
    "Você é um assistente de reuniões em português do Brasil. Sua tarefa é "
    "responder à pergunta do usuário consultando a transcrição de uma reunião "
    "por meio das ferramentas disponíveis. Use search_transcript para "
    "localizar segmentos relevantes e read_segment para lê-los na íntegra. "
    "Responda SEMPRE em português, de forma objetiva e fiel à transcrição — "
    "não invente tarefas, decisões ou nomes que não estejam nos segmentos. "
    "Quando tiver a resposta, chame return_final_answer."
)

## 4. Modelo base e registro no ART

Escolha automática pelo tamanho da GPU: T4/16GB → Qwen3-4B (smoke test); A100 → Qwen3.6-27B (denso, o alvo de produção).

In [ ]:
import importlib.util
import torch
import art
from art.local.backend import LocalBackend

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA não disponível. Ative uma GPU NVIDIA no Colab: "
        "Runtime > Change runtime type > GPU."
    )

if importlib.util.find_spec("vllm") is None:
    raise RuntimeError(
        "vLLM não está instalado neste kernel. Rode novamente a célula de setup "
        "com `%pip install -U \"openpipe-art[backend]\" \"vllm\" openai langdetect`, "
        "reinicie a sessão do Colab e execute o notebook de novo."
    )

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BASE_MODEL = "Qwen/Qwen3.6-27B" if gpu_gb > 45 else "Qwen/Qwen3-4B"
print(f"GPU: {gpu_gb:.0f}GB -> base_model: {BASE_MODEL}")

backend = LocalBackend()
model = art.TrainableModel(
    name="meeting-agent-ptbr-001",
    project="tag-ai-meeting-agent",
    base_model=BASE_MODEL,
)
await model.register(backend)
print("Modelo registrado.")


## 5. Rollout (com workaround multi-turn do Qwen 3.x)

Cada turno do assistente treina exatamente uma vez: o 1º na history principal, os subsequentes em `additional_histories` com os turnos anteriores achatados (sem `<think>`) — fiel ao que o chat template renderiza na inferência.

In [ ]:
from art.trajectories import History

try:
    from langdetect import detect
except ImportError:
    detect = None

MAX_TURNS = 6

async def rollout(model, scenario):
    client = model.openai_client()
    trajectory = art.Trajectory(
        messages_and_choices=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                f"Reunião: {scenario.title} ({scenario.date}). "
                f"A transcrição tem {len(scenario.segments)} segmentos.\n\n"
                f"Pergunta: {scenario.question}")},
        ],
        metadata={"scenario_id": scenario.id, "split": scenario.split},
    )
    final_answer = None

    for _ in range(MAX_TURNS):
        completion = await client.chat.completions.create(
            model=model.name, messages=trajectory.messages(),
            tools=TOOLS, max_tokens=1024,
        )
        choice = completion.choices[0]
        trajectory.messages_and_choices.append(choice)
        tool_calls = choice.message.tool_calls or []
        if not tool_calls:
            break
        for tc in tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if tc.function.name == "search_transcript":
                result = _search(scenario, args.get("keywords", []))
            elif tc.function.name == "read_segment":
                result = _read(scenario, int(args.get("segment_idx", -1)))
            elif tc.function.name == "return_final_answer":
                final_answer = args.get("answer", "")
                result = json.dumps({"ok": True}, ensure_ascii=False)
            else:
                result = json.dumps({"error": "ferramenta desconhecida"}, ensure_ascii=False)
            trajectory.messages_and_choices.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result})
        if final_answer is not None:
            break

    # Workaround multi-turn Qwen 3.x: cada turno treina exatamente uma vez.
    msgs = trajectory.messages_and_choices
    a_pos = [i for i, m in enumerate(msgs) if not isinstance(m, dict)]
    if len(a_pos) > 1:
        def _hist(pos):
            ctx = [m if isinstance(m, dict) else _choice_as_context(m) for m in msgs[:pos]]
            return ctx + [msgs[pos]]
        trajectory.messages_and_choices = msgs[: a_pos[0] + 1]
        trajectory.additional_histories = [
            History(messages_and_choices=_hist(p)) for p in a_pos[1:]]

    trajectory.metrics["answered"] = 1.0 if final_answer else 0.0
    if detect and final_answer:
        try:
            trajectory.metrics["is_pt"] = 1.0 if detect(final_answer) == "pt" else 0.0
        except Exception:
            trajectory.metrics["is_pt"] = 0.0
    trajectory.metadata["final_answer"] = final_answer or ""
    return trajectory

print("rollout definido")

## 6. Juiz de correção (validação) — não entra no treino

In [ ]:
judge_client = AsyncOpenAI()  # OPENAI_API_KEY
CORRECTNESS_JUDGE = "gpt-5.4"

async def judge_correctness(scenario, answer):
    if not scenario.expected:
        return 0.0
    prompt = (
        "Você avalia a resposta de um assistente de reuniões. Diga se a "
        "resposta captura corretamente os pontos da referência (mesmo com "
        "outras palavras). Responda só 'sim' ou 'não'.\n\n"
        f"Pergunta: {scenario.question}\nReferência: {scenario.expected}\n"
        f"Resposta do agente: {answer}\n")
    resp = await judge_client.chat.completions.create(
        model=CORRECTNESS_JUDGE,
        messages=[{"role": "user", "content": prompt}], max_tokens=8)
    return 1.0 if (resp.choices[0].message.content or "").strip().lower().startswith("sim") else 0.0

async def validate(model):
    scenarios = val_scenarios()
    trajs = await asyncio.gather(*(rollout(model, s) for s in scenarios))
    scores = await asyncio.gather(*(
        judge_correctness(s, t.metadata.get("final_answer", ""))
        for s, t in zip(scenarios, trajs)))
    acc = sum(scores) / len(scores) if scores else 0.0
    pt = (sum(t.metrics.get("is_pt", 0.0) for t in trajs) / len(trajs)) if trajs and detect else float("nan")
    print(f"  [validação] correção: {acc:.2%} ({len(scores)} cenários) | respostas em pt: {pt:.0%}")
    return acc

## 7. Baseline stock (antes de treinar!)

Mede o modelo SEM treino nos cenários de validação. Se já passar a barra (correção ≥ 90%, pt = 100%), o RL pode nem ser necessário — pare aqui e economize GPU.

In [ ]:
print("Baseline (modelo stock, step 0):")
baseline = await validate(model)

## 8. Treino: gather → RULER → GRPO

Com 5 cenários base, `NUM_STEPS=5` é smoke test. Com dataset expandido (célula 2), suba para 20–30.

In [ ]:
from art.rewards import ruler_score_group

RULER_JUDGE_MODEL = "openai/o4-mini"
NUM_STEPS = 5
ROLLOUTS_PER_GROUP = 4
LEARNING_RATE = 1.2e-5

for step in range(await model.get_step(), NUM_STEPS):
    print(f"\n=== Step {step} ===")
    groups = await art.gather_trajectory_groups(
        (art.TrajectoryGroup(rollout(model, sc) for _ in range(ROLLOUTS_PER_GROUP))
         for sc in train_scenarios()),
        pbar_desc="gather",
        after_each=lambda g: ruler_score_group(g, RULER_JUDGE_MODEL, swallow_exceptions=True),
    )
    await model.train(groups, config=art.TrainConfig(learning_rate=LEARNING_RATE))
    print(f"Step {step} concluído.")

print("\nTreino finalizado.")

## 9. Validação final — compare com o baseline do passo 7

In [ ]:
print(f"Baseline stock: {baseline:.2%}")
print("Depois do treino:")
final = await validate(model)
print(f"\nDelta: {final - baseline:+.2%}")

## 10. Quantizar para 4 bits (merge do LoRA + quantização)

Junta o adaptador LoRA treinado à base e salva em **4 bits** (bitsandbytes) — o formato para servir barato via vLLM/transformers. A célula localiza automaticamente o último checkpoint que o `LocalBackend` gravou em `./.art`.

⚠️ **Se der OOM:** `Ambiente de execução > Reiniciar sessão`, rode as células de instalação (seção de setup) e volte direto para cá — o checkpoint persiste em `./.art` no disco da sessão. **Atenção: o disco do Colab é efêmero** — envie ao Hugging Face (passo 11) antes de encerrar a sessão, ou o treino se perde.

In [ ]:
import gc, glob, torch

# Libera a GPU usada pelo treino/inferência do ART.
try:
    del backend
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

# Último checkpoint LoRA salvo pelo LocalBackend (dir HF-style com adapter_config.json).
ckpts = sorted(glob.glob(".art/*/models/meeting-agent-ptbr-001/checkpoints/*"))
assert ckpts, "Nenhum checkpoint encontrado — rode o treino (seção 8) antes."
CKPT = ckpts[-1]
print("Checkpoint LoRA:", CKPT)

from unsloth import FastLanguageModel

merged, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CKPT,        # o modelo base é resolvido pelo adapter_config
    max_seq_length=8192,
    load_in_4bit=True,
    dtype=None,
)

# Merge do LoRA na base + quantização 4-bit num diretório local.
merged.save_pretrained_merged(
    "meeting-agent-ptbr-4bit", tokenizer,
    save_method="merged_4bit_forced",
)
print("Salvo em ./meeting-agent-ptbr-4bit")

## 11. Enviar ao Hugging Face

Configure o secret `HF_TOKEN` no Colab (🔑) com um token **write** ([hf.co/settings/tokens](https://huggingface.co/settings/tokens)) e edite `HF_USER`. O repositório sobe como **privado** por padrão.

Publica em dois formatos:
- **(a) merged 4-bit** (safetensors/bitsandbytes) — carrega com `AutoModelForCausalLM.from_pretrained(...)` ou serve via vLLM, pronto para plugar como subagente no deep agent.
- **(b) GGUF q4_k_m** (opcional, `PUSH_GGUF=True`) — para llama.cpp / Ollama / LM Studio.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ.setdefault("HF_TOKEN", getpass("HF_TOKEN (write): "))

HF_USER = "seu-usuario"   # <-- EDITE: seu usuário/organização no Hugging Face
REPO = f"{HF_USER}/meeting-agent-ptbr-4bit"

# (a) Pesos merged + quantizados em 4 bits — formato p/ vLLM/transformers.
merged.push_to_hub_merged(
    REPO, tokenizer,
    save_method="merged_4bit_forced",
    token=os.environ["HF_TOKEN"],
    private=True,
)
print(f"Publicado: https://huggingface.co/{REPO}")

# (b) Opcional: GGUF q4_k_m (llama.cpp / Ollama / LM Studio).
PUSH_GGUF = False
if PUSH_GGUF:
    merged.push_to_hub_gguf(
        f"{REPO}-gguf", tokenizer,
        quantization_method="q4_k_m",
        token=os.environ["HF_TOKEN"],
        private=True,
    )
    print(f"Publicado: https://huggingface.co/{REPO}-gguf")

## Próximos passos

- **Delta pequeno ou negativo?** Normal com 5 cenários — expanda o dataset (célula 2, `N_EXTRA=200`) antes de concluir qualquer coisa.
- **Pipeline completo recomendado:** SFT/destilação primeiro (os `sft_pairs` do gerador), RL por cima — a pesquisa mostrou que cold-start SFT + GRPO supera RL puro, e que destilação custa ~1/10 do RL.
- **Eval matricial:** rode o `eval_matrix.py` do repositório para comparar tamanhos (4B/14B/27B) × estágios (stock/SFT/SFT+RL) e escolher o menor modelo que passa a barra — esse é o subagente mais barato viável para o Tag AI.
- **Servir:** o backend do ART expõe o checkpoint numa API compatível com OpenAI — pluga direto como subagente no deep agent (`ChatOpenAI(base_url=...)`).